# 03 — NLP sobre Movimentações Processuais

**Objetivo:** Aplicar técnicas de PLN para extrair informações estruturadas
dos textos de movimentações, com foco em:

1. Análise de frequência de movimentos
2. Classificação de resultado (procedente / improcedente / extinto)
3. Word cloud dos termos mais comuns
4. Extração de features TF-IDF

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from datajud.collectors import coletar_movimentos_dataframe
from datajud.analysis.nlp import (
    tokenizar_movimentos,
    extrair_resultado_processos,
    extrair_features_nlp,
)

sns.set_theme(style='whitegrid')

## 1. Coleta de movimentações

In [ ]:
df_mov = coletar_movimentos_dataframe(
    tribunal='tjsp',
    classe_codigo=436,
    max_docs=300,
)

print(f'Shape: {df_mov.shape}')
print(f'Processos únicos: {df_mov["numero_processo"].nunique()}')
df_mov.head()

## 2. Top movimentos mais frequentes

In [ ]:
top_movimentos = (
    df_mov['mov_nome']
    .value_counts()
    .head(20)
    .reset_index()
    .rename(columns={'mov_nome': 'movimento', 'count': 'frequencia'})
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_movimentos, y='movimento', x='frequencia', palette='Blues_r', ax=ax)
ax.set_title('Top 20 Movimentos Processuais')
plt.tight_layout()
plt.show()

top_movimentos

## 3. Tokenização e análise de termos

In [ ]:
df_tok = tokenizar_movimentos(df_mov, coluna='mov_nome')

# Frequência global de tokens
todos_tokens = [t for lista in df_tok['tokens'] for t in lista]
freq_tokens = Counter(todos_tokens)

df_freq = pd.DataFrame(freq_tokens.most_common(30), columns=['termo', 'frequencia'])

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=df_freq, y='termo', x='frequencia', palette='viridis', ax=ax)
ax.set_title('Top 30 Termos nas Movimentações')
plt.tight_layout()
plt.show()

## 4. Classificação de resultado processual

In [ ]:
df_resultado = extrair_resultado_processos(df_mov)

print(f'Processos com resultado identificado: {len(df_resultado)}')

if not df_resultado.empty:
    dist = df_resultado['resultado'].value_counts()

    fig, ax = plt.subplots(figsize=(7, 4))
    dist.plot(kind='bar', color=['#2ecc71', '#e74c3c', '#f39c12'], ax=ax)
    ax.set_title('Distribuição de Resultados Processuais')
    ax.set_xlabel('Resultado')
    ax.set_ylabel('Quantidade')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    df_resultado.head()

## 5. Extração de features TF-IDF

In [ ]:
matriz, vetorizador, df_agg = extrair_features_nlp(
    df_mov,
    coluna_texto='mov_nome',
    max_features=300,
    agrupar_por_processo=True,
)

print(f'Matriz TF-IDF: {matriz.shape}')
print(f'Exemplos de features: {vetorizador.get_feature_names_out()[:20]}')

## 6. Salvar artefatos

In [ ]:
import pickle, scipy.sparse as sp

df_mov.to_parquet('../data/processed/movimentos_tjsp.parquet', index=False)
df_resultado.to_parquet('../data/processed/resultados_tjsp.parquet', index=False)

sp.save_npz('../data/processed/tfidf_matrix.npz', matriz)
with open('../data/processed/tfidf_vetorizador.pkl', 'wb') as f:
    pickle.dump(vetorizador, f)

print('Artefatos salvos em data/processed/')